# Real daily SPY data audit

Validate the real daily OHLCV source before evaluating any limit-order rule. This notebook contains no synthetic or monthly Shiller path.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

from retail_sp500.daily_data import daily_data_summary, load_or_fetch_twelve_data_daily

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.6f}")

## Load validated daily sessions

The first run requires `TWELVE_DATA_API_KEY`; subsequent runs use the ignored local CSV cache. Set `REFRESH = True` to update it.

In [ ]:
SYMBOL = "SPY"
START_DATE = "2007-06-01"
CACHE_PATH = Path("data/processed/spy_daily_1day.csv")
REFRESH = False

daily = load_or_fetch_twelve_data_daily(
    os.getenv("TWELVE_DATA_API_KEY"),
    cache_path=CACHE_PATH,
    refresh=REFRESH,
    symbol=SYMBOL,
    start_date=START_DATE,
)

source = daily_data_summary(daily, symbol=SYMBOL)
print(source)
assert source["interval"] == "1day"
assert daily.index.max() <= pd.Timestamp.today().normalize()
daily.tail()

## Price history and session diagnostics

In [ ]:
diagnostics = daily.assign(
    close_return=daily["close"].pct_change(),
    overnight_gap=daily["open"] / daily["close"].shift(1) - 1.0,
    intraday_low_from_previous_close=daily["low"] / daily["close"].shift(1) - 1.0,
    intraday_high_from_previous_close=daily["high"] / daily["close"].shift(1) - 1.0,
).dropna()

diagnostics[[
    "close_return",
    "overnight_gap",
    "intraday_low_from_previous_close",
    "intraday_high_from_previous_close",
]].describe(percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99])

In [ ]:
px.line(
    daily.reset_index(),
    x="date",
    y="close",
    log_y=True,
    title=f"{SYMBOL} real daily close",
).show()

## How far below the prior close does SPY trade?

This distribution is descriptive only. It does not account for missed rallies or delayed capital deployment.

In [ ]:
dip_below_previous_close = (
    1.0 - diagnostics["low"] / diagnostics["close"].shift(1)
).dropna()

dip_below_previous_close.quantile(
    [0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
).rename("discount_touched")

In [ ]:
px.histogram(
    dip_below_previous_close.clip(lower=-0.03, upper=0.08),
    nbins=120,
    title="Daily low relative to the preceding close",
    labels={"value": "1 - low / previous close"},
).show()